# Project 2: Predict California House Prices with Linear Regression

**Goal:** predict the median house value of a neighbourhood (a 'block group' from the 1990 US census) using its income level, house age, rooms, population and location.

**Dataset:** California Housing, about 20,640 rows. It is built into scikit-learn, so it loads with one line and there is no CSV to download or upload.

**Label (what we predict):** `MedHouseVal`, in units of 100,000 dollars. So 2.5 means 250,000 dollars.

**Steps:** 1) Load, 2) Explore, 3) Visualize, 4) Split, 5) Train, 6) Evaluate, 7) Read the coefficients, 8) Check the errors (residuals), 9) Improve the model, 10) Predict for a new house.

Run the cells from top to bottom (Shift + Enter).

## Step 1: Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame
df.head()

**What the columns mean**

| Column | Meaning |
|---|---|
| MedInc | Median income in the block (in 10,000 dollars) |
| HouseAge | Median age of the houses |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | People living in the block |
| AveOccup | Average people per household |
| Latitude, Longitude | Location of the block |
| MedHouseVal | **Label:** median house value (in 100,000 dollars) |

## Step 2: Explore the data

Check the size, missing values and basic statistics. Look at the `max` row: some columns (like `AveRooms`, `AveOccup`) have extreme outliers.

In [ ]:
print('Rows, columns:', df.shape)
print()
print('Missing values per column:')
print(df.isnull().sum())
print()
df.describe().round(2)

## Step 3: Visualize

Find which features move together with the price. The heatmap shows correlation: close to +1 or -1 means a strong relationship.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['MedHouseVal'], bins=40, ax=axes[0])
axes[0].set_title('Distribution of house value')

sns.scatterplot(x='MedInc', y='MedHouseVal', data=df.sample(3000, random_state=1), alpha=0.4, ax=axes[1])
axes[1].set_title('Income vs house value')

sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[2], cbar=False, annot_kws={'size': 7})
axes[2].set_title('Correlation between columns')

plt.tight_layout()
plt.show()

print('Correlation of each feature with the price:')
print(df.corr()['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False).round(2))

**What you should notice:**
- `MedInc` (income) has the strongest link to the price.
- The price histogram has a spike at the right end. The original data **caps** house values at 5.0 (500,000 dollars), so many expensive blocks are all recorded as exactly 5.0. This hurts a linear model, and we will handle it in Step 9.

## Step 4: Split into training and test data

80% to train the model, 20% held back to test it honestly. All the columns are already numbers, so no encoding is needed.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('MedHouseVal', axis=1)   # features
y = df['MedHouseVal']                # label

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Training rows:', len(X_train), '| Test rows:', len(X_test))

## Step 5: Train the baseline model

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
print('Training finished.')

## Step 6: Evaluate on the unseen test data

Remember the label is in units of 100,000 dollars, so a MAE of 0.5 means the prediction is off by about 50,000 dollars on average.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print('MAE  :', round(mae, 3), ' (about', int(mae * 100000), 'dollars off on average)')
print('RMSE :', round(rmse, 3))
print('R2 (test) :', round(r2, 4))
print('R2 (train):', round(model.score(X_train, y_train), 4))

plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.3, s=10)
plt.plot([0, 5], [0, 5], 'r--', label='Perfect prediction')
plt.xlabel('Actual value')
plt.ylabel('Predicted value')
plt.title('Actual vs predicted (baseline)')
plt.legend()
plt.show()

**How to read this:** you should see a test R2 of roughly 0.57 to 0.60, meaning the model explains a bit under 60% of the variation in house prices. That is modest, and it is a good lesson: a plain straight-line model has limits on messy real-world data. Train and test R2 are close, so this is **underfitting** (the model is too simple), not overfitting. The horizontal row of dots at the top of the plot is the capped 5.0 values.

## Step 7: Read the coefficients

The raw coefficients cannot be compared fairly because the features have different scales (income is 0 to 15, population is in the thousands). So we first **standardize** the features (mean 0, spread 1), then train again. Now a bigger coefficient really means a stronger effect.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)

model_s = LinearRegression().fit(X_train_s, y_train)

coef = pd.DataFrame({'feature': X.columns, 'coefficient': model_s.coef_})
coef = coef.sort_values('coefficient', ascending=False)
print(coef.round(3).to_string(index=False))

plt.figure(figsize=(8, 4))
sns.barplot(x='coefficient', y='feature', data=coef)
plt.title('Effect of each feature (standardized)')
plt.show()

**What you should notice:**
- `MedInc` has a large positive effect: richer areas have pricier houses.
- `Latitude` and `Longitude` have large negative effects: moving north or east (away from the southern coast) lowers the price.
- `AveRooms` comes out negative and `AveBedrms` positive, which seems odd. This happens because those two columns are strongly correlated with each other (called **multicollinearity**), so the model splits their effect in a confusing way. Do not read single coefficients too literally when features overlap.

## Step 8: Check the errors (residuals)

A residual is `actual - predicted`. For a good linear model, residuals should be scattered randomly around 0 with no pattern. A visible pattern means the model is missing something.

In [ ]:
residuals = y_test - pred

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(pred, residuals, alpha=0.3, s=10)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted value')
axes[0].set_ylabel('Residual (actual - predicted)')
axes[0].set_title('Residuals vs predictions')

sns.histplot(residuals, bins=50, kde=True, ax=axes[1])
axes[1].set_title('Distribution of residuals')

plt.tight_layout()
plt.show()

You should see a diagonal band of dots in the left plot. That band comes from the capped 5.0 prices, and it tells us the data (not just the model) is causing trouble. Let's fix that next.

## Step 9: Improve the model

Three fixes, each based on what we found above:

1. **Remove the capped rows** (`MedHouseVal >= 5`), because their true value is unknown.
2. **Clip extreme outliers** in `AveRooms`, `AveBedrms`, `AveOccup` and `Population` at the 99th percentile.
3. **Add engineered features:** `RoomsPerBedroom` and `LogPopulation` (a log tames very large populations).

We compare the old and new model using **5-fold cross-validation**, which gives a more reliable score than a single split.

In [ ]:
from sklearn.model_selection import cross_val_score

# 1) remove capped rows
df2 = df[df['MedHouseVal'] < 5].copy()
print('Rows after removing capped prices:', len(df2), '(removed', len(df) - len(df2), ')')

# 2) clip outliers at the 99th percentile
for col in ['AveRooms', 'AveBedrms', 'AveOccup', 'Population']:
    df2[col] = df2[col].clip(upper=df2[col].quantile(0.99))

# 3) engineered features
df2['RoomsPerBedroom'] = df2['AveRooms'] / df2['AveBedrms']
df2['LogPopulation'] = np.log1p(df2['Population'])

X2 = df2.drop('MedHouseVal', axis=1)
y2 = df2['MedHouseVal']

# baseline vs improved, both with 5-fold cross-validation
base_cv = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')
new_cv = cross_val_score(LinearRegression(), X2, y2, cv=5, scoring='r2')

print()
print('Baseline model  mean R2 (5-fold):', round(base_cv.mean(), 4))
print('Improved model  mean R2 (5-fold):', round(new_cv.mean(), 4))
print()
print('Note: the improved model is scored on a cleaner dataset, so treat the')
print('comparison as a guide, not a perfectly fair race.')

The improved model should score somewhat higher. Even so, linear regression will not get far past about 0.6 to 0.65 here, because house prices depend on **location in a non-linear way** (a house on the coast in a good city is worth far more than a straight line suggests). Models like random forests handle this much better, which is a natural next step after you master linear regression.

## Step 10: Predict for a new house

We train the improved model on all the cleaned data and predict for a made-up block. Change the numbers and re-run.

In [ ]:
final_model = LinearRegression().fit(X2, y2)

new_house = {
    'MedInc': 5.0,        # median income of 50,000 dollars
    'HouseAge': 20,
    'AveRooms': 5.5,
    'AveBedrms': 1.1,
    'Population': 1500,
    'AveOccup': 3.0,
    'Latitude': 34.0,     # around Los Angeles
    'Longitude': -118.3,
}
new_house['RoomsPerBedroom'] = new_house['AveRooms'] / new_house['AveBedrms']
new_house['LogPopulation'] = np.log1p(new_house['Population'])

new_X = pd.DataFrame([new_house])[X2.columns]
value = final_model.predict(new_X)[0]
print('Predicted median house value:', round(value, 2), '->', int(value * 100000), 'dollars')

## Practice ideas

1. Change `test_size` or `random_state` in Step 4 and see how R2 moves.
2. Train a model using only `MedInc` (simple linear regression) and plot the fitted line over the scatter plot.
3. Try `Ridge` and `Lasso` from `sklearn.linear_model` and compare their scores with plain linear regression.
4. Try `RandomForestRegressor` from `sklearn.ensemble` on the same split and compare R2. Notice how much location-based patterns matter.
5. Plot a map: `plt.scatter(df['Longitude'], df['Latitude'], c=df['MedHouseVal'], s=3)` to see the price pattern across California.